# $$ \textbf{Aprendizaje Supervizado}$$
# $$\text{Tarea 4}$$
# $$\text{Daniel Rojo Mata}$$

# **Ejercicio 2: Representación TF-IDF y Clasificación de Textos**

Utilizando la base de datos de **20 Newsgroups**, calcula la representación de **bolsa de palabras con TF-IDF**. Posteriormente, realiza las siguientes tareas:

### Tareas a realizar

1. **Comparación de kernels en SVM**  
   Entrena y compara el rendimiento de un clasificador **SVM** utilizando los siguientes kernels:
   - Kernel **lineal**
   - Kernel **RBF**
      Kernel **coseno**

2. **Demostración teórica del kernel coseno como núcleo de Mercer**  
   Demuestra que el kernel coseno:

   $$
   k(x, x') = \frac{x^\top x'}{\|x\|_2 \cdot \|x'\|_2}
   $$

   cumple con las condiciones del **teorema de Mercer**: es simétrico y definido positivo.

> El kernel coseno es comúnmente utilizado en tareas de clasificación de texto, ya que considera la dirección del vector más que su magnitud, lo cual es adecuado para representar documentos en espacios de alta dimensión.

# **Librerías**

In [9]:
# Importa el conjunto de datos "20 Newsgroups", un dataset clásico de clasificación de texto
from sklearn.datasets import fetch_20newsgroups

# Importa el vectorizador TF-IDF, que convierte texto en una matriz de características numéricas
from sklearn.feature_extraction.text import TfidfVectorizer

# Importa el clasificador de Máquinas de Vectores de Soporte (SVM)
from sklearn.svm import SVC

# Importa la métrica de precisión para evaluar el rendimiento del modelo
from sklearn.metrics import accuracy_score

# Importa la función para calcular la similitud coseno entre vectores
from sklearn.metrics.pairwise import cosine_similarity

# Permite crear pipelines de procesamiento y modelado
from sklearn.pipeline import make_pipeline

# Normaliza vectores para que tengan norma 1 (útil en combinación con similitud coseno)
from sklearn.preprocessing import Normalizer

# Importa NumPy para operaciones numéricas
import numpy as np

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# **Data**

## Carga y vectorización de texto con TF-IDF

En esta etapa del proyecto, se trabaja con el conjunto de datos **20 Newsgroups**, uno de los corpus más conocidos para tareas de clasificación de texto. Este dataset contiene miles de documentos distribuidos en 20 categorías temáticas, como ciencia, deportes, política, religión, entre otras.

Como parte del preprocesamiento, se eliminan elementos estructurales como encabezados, pies de página y citas, ya que pueden introducir sesgos o redundancias que dificultan la generalización del modelo. Luego, se transforman los documentos en una representación numérica utilizando la técnica **TF-IDF (Term Frequency - Inverse Document Frequency)**.

Esta técnica asigna un peso a cada palabra en un documento que refleja su importancia relativa dentro del conjunto de datos: palabras frecuentes en un documento pero poco comunes en el resto del corpus reciben un mayor valor, permitiendo capturar mejor la información discriminativa de cada texto.

El objetivo de esta fase es obtener una matriz de características que pueda ser utilizada por algoritmos de aprendizaje automático, como SVM, para realizar tareas de clasificación. Esta representación vectorial es esencial, ya que los modelos no pueden operar directamente sobre texto en bruto.

In [13]:
# Se definen las categorías a usar; si es None, se usan las 20 categorías completas del dataset
categorias_deseadas = None

# Carga el conjunto de datos de entrenamiento, eliminando encabezados, pies y citas para evitar sesgos
conjunto_entrenamiento = fetch_20newsgroups(
    subset='train',
    categories=categorias_deseadas,
    remove=('headers', 'footers', 'quotes')
)

# Carga el conjunto de datos de prueba con las mismas condiciones que el de entrenamiento
conjunto_prueba = fetch_20newsgroups(
    subset='test',
    categories=categorias_deseadas,
    remove=('headers', 'footers', 'quotes')
)

# Crea un vectorizador TF-IDF para convertir texto a vectores numéricos
# stop_words='english' elimina palabras comunes en inglés
# max_df=0.5 ignora términos que aparecen en más del 50% de los documentos
vectorizador_tfidf = TfidfVectorizer(stop_words='english', max_df=0.5)

# Aplica el vectorizador al conjunto de entrenamiento y construye la matriz de características
matriz_tfidf_entrenamiento = vectorizador_tfidf.fit_transform(conjunto_entrenamiento.data)

# Transforma los textos del conjunto de prueba usando el mismo vectorizador entrenado
matriz_tfidf_prueba = vectorizador_tfidf.transform(conjunto_prueba.data)

# Extrae las etiquetas de clase correspondientes a cada documento de entrenamiento
etiquetas_entrenamiento = conjunto_entrenamiento.target

# Extrae las etiquetas de clase correspondientes a cada documento de prueba
etiquetas_prueba = conjunto_prueba.target

# **Entrenamiento**

## Entrenamiento de modelos SVM con diferentes núcleos (kernels)

En esta sección se comparan distintos enfoques para entrenar modelos de clasificación de texto usando **Máquinas de Vectores de Soporte (SVM)**. Esta técnica es muy utilizada por su capacidad de generar fronteras de decisión óptimas incluso en espacios de alta dimensión, como los generados por representaciones TF-IDF.

Se prueban tres variantes de SVM:

- **Kernel lineal**: Utiliza el producto punto entre vectores. Es eficiente y adecuado cuando los datos son aproximadamente separables de manera lineal en el espacio de características. Funciona bien en muchos problemas de texto por la alta dimensionalidad y dispersión del espacio vectorial.

- **Kernel RBF (Radial Basis Function)**: Permite capturar relaciones no lineales complejas al proyectar los datos en un espacio de mayor dimensión. Es útil cuando la separación lineal no es posible en el espacio original.

- **Kernel basado en similitud coseno**: En lugar de un kernel estándar, calculamos explícitamente la matriz de similitud coseno entre documentos, una medida angular que se adapta muy bien a datos textuales. Este kernel no viene predefinido en SVM, por lo que lo integramos como una matriz **precomputada**, lo cual permite a SVM tratar directamente la similitud entre documentos como función de decisión.

Cada variante fue evaluada usando la métrica de **exactitud**, que representa el porcentaje de etiquetas correctamente clasificadas en el conjunto de prueba. Esto nos permite comparar el desempeño de cada enfoque y elegir el que mejor se adapta a nuestro problema de clasificación de texto.

In [14]:
def entrenar_y_evaluar_svm(X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, kernel='linear', precomputed=False):
    """
    Entrena un modelo SVM con el kernel especificado y calcula la exactitud.
    
    Si `precomputed` es True, se asume que X_entrenamiento y X_prueba son matrices de kernel.
    """
    modelo = SVC(kernel=kernel)
    modelo.fit(X_entrenamiento, y_entrenamiento)
    predicciones = modelo.predict(X_prueba)
    exactitud = accuracy_score(y_prueba, predicciones)
    return exactitud

# 1. Kernel lineal
exactitud_lineal = entrenar_y_evaluar_svm(
    matriz_tfidf_entrenamiento,
    etiquetas_entrenamiento,
    matriz_tfidf_prueba,
    etiquetas_prueba,
    kernel='linear'
)

# 2. Kernel RBF
exactitud_rbf = entrenar_y_evaluar_svm(
    matriz_tfidf_entrenamiento,
    etiquetas_entrenamiento,
    matriz_tfidf_prueba,
    etiquetas_prueba,
    kernel='rbf'
)

# 3. Kernel de similitud coseno (requiere precomputación y normalización)
normalizador = Normalizer()
X_entrenamiento_normalizado = normalizador.fit_transform(matriz_tfidf_entrenamiento)
X_prueba_normalizado = normalizador.transform(matriz_tfidf_prueba)

# Matrices de kernel basadas en la similitud coseno
kernel_coseno_entrenamiento = cosine_similarity(X_entrenamiento_normalizado)
kernel_coseno_prueba = cosine_similarity(X_prueba_normalizado, X_entrenamiento_normalizado)

# Entrenar modelo con kernel precomputado
exactitud_coseno = entrenar_y_evaluar_svm(
    kernel_coseno_entrenamiento,
    etiquetas_entrenamiento,
    kernel_coseno_prueba,
    etiquetas_prueba,
    kernel='precomputed'
)

# Mostrar resultados
print("Exactitud SVM Lineal:", exactitud_lineal)
print("Exactitud SVM RBF:", exactitud_rbf)
print("Exactitud SVM Coseno:", exactitud_coseno)

Exactitud SVM Lineal: 0.6755177907594264
Exactitud SVM RBF: 0.6789697291556027
Exactitud SVM Coseno: 0.6755177907594264


## Tabla de Resultados de Exactitud

| Modelo SVM         | Tipo de Kernel             | Exactitud (%) |
|--------------------|----------------------------|----------------|
| SVM Lineal         | `linear`                   | 67.55%         |
| SVM con RBF        | `rbf`                      | 67.90%         |
| SVM con Coseno     | `precomputed` (coseno)     | 67.55%         |

---

## **Conclusión**

Los resultados muestran que los tres modelos tienen un desempeño **muy similar**, con una **ligera ventaja para el kernel RBF**, que alcanza una exactitud del 67.90%. Esto sugiere que, aunque los datos no son perfectamente separables de forma lineal, la complejidad adicional del RBF no aporta una mejora significativa en este caso específico.

El kernel de similitud coseno logra la misma exactitud que el kernel lineal, lo cual es esperable dado que ambos capturan relaciones basadas en la dirección de los vectores en un espacio normalizado. Este resultado refuerza una observación común en tareas de clasificación de texto: **los modelos lineales suelen ser verdaderamente efectivos** debido a la alta dimensionalidad y dispersión del espacio TF-IDF.

En resumen: si se busca eficiencia y simplicidad, el **SVM lineal** sigue siendo una opción robusta. Si se desea explorar estructuras no lineales más complejas, el **kernel RBF** puede aportar ligeras mejoras, aunque en este caso no fue significativo.

### Prueba: El kernel coseno cumple con las condiciones de Mercer

Sea el kernel coseno definido como:

$$
k(\mathbf{x}, \mathbf{x'}) = \frac{\mathbf{x}^\top \mathbf{x'}}{\|\mathbf{x}\|_2 \cdot \|\mathbf{x'}\|_2}
$$

Queremos demostrar que este kernel cumple con las condiciones del **teorema de Mercer**, es decir, que genera una matriz de Gram simétrica y semidefinida positiva.

#### Demostración

Sea $k(\mathbf{x}, \mathbf{x'})$ el kernel coseno, y sea $\mathbf{K}$ su matriz de Gram, con entradas:

$$
K_{ij} = k(\mathbf{x}^{(i)}, \mathbf{x}^{(j)}).
$$

Supongamos que todos los vectores han sido **normalizados**, es decir:

$$
\|\mathbf{x}^{(i)}\|_2 = 1 \quad \text{para todo } i.
$$

Entonces, el kernel se simplifica a:

$$
k(\mathbf{x}^{(i)}, \mathbf{x}^{(j)}) = \mathbf{x}^{(i)\top} \mathbf{x}^{(j)},
$$

es decir, el **producto punto estándar**. Bajo esta suposición, la matriz $\mathbf{K}$ coincide con la **matriz de Gram** construida a partir de los vectores normalizados.

Como es sabido, cualquier matriz de Gram construida mediante productos punto es:

- Simétrica  
- Semidefinida positiva

Por lo tanto, el **kernel coseno** cumple con las condiciones del teorema de **Mercer**.

**Q.E.D.**